### imports

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import cv2


### network architecture

In [2]:
class Residual_block(torch.nn.Module):
    def __init__(self,channels):
        super().__init__()

        self.conv1=torch.nn.Conv2d(in_channels=channels,out_channels=channels,kernel_size=3,padding=1)
        self.relu=torch.nn.ReLU()
        self.conv2=torch.nn.Conv2d(in_channels=channels,out_channels=channels,kernel_size=3,padding=1)
        
    def forward(self,x):
        y=self.conv1(x)
        y=self.relu(y)
        y=self.conv2(y)+x
        y=self.relu(y)
        return y
        

In [3]:
class Base_model(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=torch.nn.Conv2d(in_channels=3,out_channels=16,kernel_size=3)
        self.relu=torch.nn.ReLU()
        self.maxpool=torch.nn.MaxPool2d(kernel_size=2)
        self.conv2=torch.nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3)
        self.fc1=torch.nn.Linear(in_features=1152,out_features=1470)
        
    def forward(self, x):
        y=self.conv1(x)
        y=self.relu(y)
        y=self.maxpool(y)
        y=self.conv2(y)
        y=self.relu(y)
        y=self.maxpool(y)
        y=self.fc1(y)
        y=y.view(-1,7,7,30)
        
        return y

This number is the most likely to crash your code if you change your input image size without recalculating it. It is strictly determined by how your image shrinks as it moves through the layers.Let's trace a standard $32 \times 32$ pixel input image through your specific model:Input: $32 \times 32$conv1 (3x3 kernel): A 3x3 sliding window shaves 1 pixel off every edge. $32 - 2 = 30$. The image is now $30 \times 30$.maxpool (2x2 kernel): Halves the dimensions. $30 / 2 = 15$. The image is now $15 \times 15$.conv2 (3x3 kernel): Shaves 1 pixel off every edge. $15 - 2 = 13$. The image is now $13 \times 13$.maxpool (2x2 kernel): Halves the dimensions (PyTorch rounds down). $13 // 2 = 6$. The image is now $6 \times 6$.At the end of your feature extractor, you have 32 channels (from conv2), and each channel is a $6 \times 6$ grid.To flatten this into a 1D list for the linear layer: $32 \times 6 \times 6 = 1152$

4. The YOLO Output Logic: 1470 and .view(-1, 7, 7, 30)These numbers are determined by the YOLOv1 architecture rules. The goal is to predict a $7 \times 7$ grid, where each cell predicts 2 boxes (5 values each) and 20 classes.30: The number of values needed per grid cell: $(2 \text{ boxes} \times 5 \text{ values}) + 20 \text{ classes} = 30$.1470: The total flat number of predictions needed for the whole image: $7 \times 7 \times 30 = 1470$.-1: In PyTorch's .view(), passing -1 tells PyTorch to automatically figure out that dimension based on whatever data is left over. We put it in the first position to safely represent the Batch Size, so the code won't break whether you feed it 1 image or a batch of 64 images.

Our current model outputs a simple 1D list of 10 numbers (the probabilities for 10 classes). YOLO, however, predicts multiple bounding boxes and class probabilities all at once. To do this, it conceptually divides the image into an $S \times S$ grid. For each grid cell, it predicts:$B$ bounding boxes. Each box has 5 values: x, y, width, height, and a confidence score.$C$ class probabilities.Because the final linear layer still needs to output a 1D vector, YOLO flattens all of these predictions into one long list. The total number of outputs required is calculated using this formula:$$S \times S \times (B \times 5 + C)$$In the original YOLOv1 paper, they used a $7 \times 7$ grid ($S=7$), predicted 2 bounding boxes per cell ($B=2$), and classified 20 different objects ($C=20$).

u get 1470

These numbers are determined by the YOLOv1 architecture rules. The goal is to predict a $7 \times 7$ grid, where each cell predicts 2 boxes (5 values each) and 20 classes.30: The number of values needed per grid cell: $(2 \text{ boxes} \times 5 \text{ values}) + 20 \text{ classes} = 30$.1470: The total flat number of predictions needed for the whole image: $7 \times 7 \times 30 = 1470$.-1: In PyTorch's .view(), passing -1 tells PyTorch to automatically figure out that dimension based on whatever data is left over. We put it in the first position to safely represent the Batch Size, so the code won't break whether you feed it 1 image or a batch of 64 images.

note:(62 $\rightarrow$ 31 $\rightarrow$ 29 $\rightarrow$ 14.5)

pytorch will always round down so it will be 14 
if input is 64x64 then output will be 6272 in fcq input_dim 

### loss fn working


In [4]:
# class_predictiion=predictions[:,:,:,10:]

#target_classes = targets[:,:,:,10:]

# u extract and compair the class prediction with target classes
#mse = torch.nn.MSELoss()
# classification_loss = mse(class_prediction, target_classes)

# box_1=predictions[:,:,:,0:4]
# box_2=predictions[:,:,:,4:9]

### darknet-style

using resudial block


In [5]:
class DeepYOLOBackbone(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=torch.nn.Conv2d(in_channels=3,out_channels=64,kernel_size=3)
        self.relu=torch.nn.ReLU()
        self.res_blocks=torch.nn.Sequential(Residual_block(64),Residual_block(64),Residual_block(64))

    def forward(self,x):
        x=self.conv1(x)
        x=self.relu(x)
        x=self.res_blocks(x)

        return x


### data class 

In [ ]:
from torch.utils.data import  Dataset
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2


class YOLODataset(Dataset):
    def __init__(self,image_paths,labels):
        self.image_paths = image_paths
        self.labels = labels
        self.transforms=A.Compose([ToTensorV2()])


    def __getitem__(self,index):

        image=cv2.imread(self.image_paths[index])
        image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        label=self.labels[index]
        transformed_image=self.transforms(image=image)
        image_tensor=transformed_image["image"]

        return(image_tensor,labels)



    def __len__(self):
        return len(self.image_paths)



### dataloader


In [11]:
import torch.optim as optim


model = DeepYOLOBackbone()

In [12]:
model

DeepYOLOBackbone(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (res_blocks): Sequential(
    (0): Residual_block(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (relu): ReLU()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (1): Residual_block(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (relu): ReLU()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (2): Residual_block(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (relu): ReLU()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
  )
)

In [13]:
optimizer=torch.optim.Adam(model.parameters(), lr=0.001)